In [ ]:
from datasets import load_dataset

ds = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="likes.parquet")


In [ ]:
df = ds["train"].to_pandas()
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.columns)

In [ ]:
print(df.uid.nunique(), df.item_id.nunique())
print(df.timestamp.min(), df.timestamp.max())
print(df.is_organic.mean())
print(df.groupby("uid").size().describe())
print(df.duplicated(["uid", "item_id"]).mean())  # повторные лайки одного трека
listens = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="listens.parquet")["train"].to_pandas()
print(listens.shape); print(listens.head()); print(listens.dtypes)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
users = rng.choice(listens.uid.unique(), 2000, replace=False)

L = listens[listens.uid.isin(users)].copy()
K = df[df.uid.isin(users)]            # лайки
print(L.shape, K.shape)
del listens                           # освободить память


In [ ]:
like_ts = (K.groupby(["uid", "item_id"]).timestamp.min()
             .rename("like_ts").reset_index())
L = L.merge(like_ts, on=["uid", "item_id"], how="left")

dt = L.like_ts.astype("float") - L.timestamp.astype("float")
L["label"] = ((dt >= 0) & (dt <= 3600)).astype("int8")   # окно 1 час, потом подкрутим
L = L.drop(columns="like_ts")
print(L.label.mean())

In [ ]:
T = L.timestamp
t_test = T.max() - 14*86400
t_val  = t_test - 14*86400

train = L[T < t_val]
val   = L[(T >= t_val) & (T < t_test)]
test  = L[T >= t_test]
print(len(train), len(val), len(test))

In [ ]:
import time
import pandas as pd
import numpy as np

DAY = 86400
WINDOWS = (1, 7, 30, None)

def build(cand, cutoff, windows=WINDOWS):
    HL = L[L.timestamp < cutoff]
    HK = K[K.timestamp < cutoff]

    # проверка на утечку: история строго до среза, кандидаты строго после
    assert HL.timestamp.max() < cutoff and HK.timestamp.max() < cutoff
    assert cand.timestamp.min() >= cutoff

    X = cand[["uid", "item_id", "is_organic", "label"]].copy()

    for w in windows:
        lo = 0 if w is None else cutoff - w * DAY
        l = HL[HL.timestamp >= lo]
        k = HK[HK.timestamp >= lo]
        wn = "all" if w is None else f"{w}d"

        for ent, keys in [("u", ["uid"]), ("i", ["item_id"]), ("ui", ["uid", "item_id"])]:
            for ev, h in [("listens", l), ("likes", k)]:
                col = f"{ent}_{ev}_{wn}"
                c = h.groupby(keys).size().rename(col).reset_index()
                X = X.merge(c, on=keys, how="left")
                X[col] = X[col].fillna(0)

        for ent in ["u", "i"]:
            X[f"{ent}_cr_{wn}"] = X[f"{ent}_likes_{wn}"] / (X[f"{ent}_listens_{wn}"] + 1)

        # средняя доля прослушивания в прошлом
        for ent, keys in [("u", "uid"), ("i", "item_id")]:
            col = f"{ent}_played_{wn}"
            c = l.groupby(keys).played_ratio_pct.mean().rename(col).reset_index()
            X = X.merge(c, on=keys, how="left")   # NaN оставляем: бустинг умеет с ними работать
    return X

def window(a, b):
    return L[(L.timestamp >= a) & (L.timestamp < b)]

t0 = time.time()
c_tr = t_val - 14 * DAY
Xtr = build(window(c_tr, t_val),         c_tr);   print("train", Xtr.shape, time.time() - t0)
Xva = build(window(t_val, t_test),       t_val);  print("val",   Xva.shape, time.time() - t0)
Xte = build(window(t_test, T.max() + 1), t_test); print("test",  Xte.shape, time.time() - t0)
print(Xtr.label.mean(), Xva.label.mean(), Xte.label.mean())

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss

def rank_metrics(X, score, k=10):
    d = pd.DataFrame({"uid": X.uid.values, "y": X.label.values, "s": np.asarray(score, dtype="float64")})
    d["s"] += np.random.default_rng(0).random(len(d)) * 1e-9      # случайный разбор ничьих
    d = d.sort_values(["uid", "s"], ascending=[True, False])
    d["r"] = d.groupby("uid").cumcount() + 1

    npos = d.groupby("uid").y.sum()
    npos = npos[npos > 0]                                          # только юзеры с позитивом
    top = d[(d.r <= k) & d.uid.isin(npos.index)]

    dcg = (top.y / np.log2(top.r + 1)).groupby(top.uid).sum().reindex(npos.index).fillna(0)
    ideal = np.cumsum(1 / np.log2(np.arange(2, k + 2)))
    idcg = pd.Series(ideal[np.minimum(npos.values, k).astype(int) - 1], index=npos.index)
    hits = top.y.groupby(top.uid).sum().reindex(npos.index).fillna(0)
    return {f"ndcg@{k}": (dcg / idcg).mean(), f"recall@{k}": (hits / npos).mean()}

def report(name, X, score):
    r = {"AUC": roc_auc_score(X.label, score)}
    r.update(rank_metrics(X, score))
    print(f"{name:28s}", {k: round(v, 4) for k, v in r.items()})
    return r

rng = np.random.default_rng(1)
res = {}
for nm, X in [("val", Xva), ("test", Xte)]:
    res[f"random/{nm}"]  = report(f"random [{nm}]",      X, rng.random(len(X)))
    res[f"popular/{nm}"] = report(f"top-popular [{nm}]", X, X.i_likes_all.values)

    # pip install lightgbm
import lightgbm as lgb

def fit_eval(name, feats, params=None):
    p = dict(n_estimators=1000, learning_rate=0.05, num_leaves=63,
             subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
    p.update(params or {})
    m = lgb.LGBMClassifier(**p)
    m.fit(Xtr[feats], Xtr.label, eval_set=[(Xva[feats], Xva.label)],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    out = {}
    for nm, X in [("val", Xva), ("test", Xte)]:
        out[nm] = report(f"{name} [{nm}]", X, m.predict_proba(X[feats])[:, 1])
        out[nm]["logloss"] = log_loss(X.label, m.predict_proba(X[feats])[:, 1])
    return m, out

base = ["is_organic"] + [c for c in Xtr.columns if c.endswith("_all") and c != "uid"]
m_min, r_min = fit_eval("boosting minimal", base)

In [ ]:
Xtr, Xva, Xte = [X[X.ui_likes_all == 0].reset_index(drop=True) for X in (Xtr, Xva, Xte)]
print(len(Xtr), len(Xva), len(Xte), Xtr.label.mean(), Xva.label.mean(), Xte.label.mean())

rng = np.random.default_rng(1)
for nm, X in [("val", Xva), ("test", Xte)]:
    report(f"random [{nm}]", X, rng.random(len(X)))
    report(f"top-popular [{nm}]", X, X.i_likes_all.values)

base = ["is_organic"] + [c for c in Xtr.columns if c.endswith("_all")]
m_min, r_min = fit_eval("boosting minimal", base)

all_feats = [c for c in Xtr.columns if c not in ("uid", "item_id", "label")]
m_full, r_full = fit_eval("boosting full (manual)", all_feats)

imp = pd.Series(m_full.booster_.feature_importance("gain"), index=all_feats).sort_values(ascending=False)
print(imp.head(15))

In [ ]:
def build_train(n_cuts=4, step=7):
    parts = []
    for k in range(1, n_cuts + 1):
        c = t_val - k * step * DAY
        cand = window(c, c + step * DAY)
        X = build(cand, c)
        parts.append(X[X.ui_likes_all == 0])
    return pd.concat(parts, ignore_index=True)

Xtr = build_train()
print(Xtr.shape, Xtr.label.mean())

m_full4, r_full4 = fit_eval("boosting full, 4 cuts", all_feats)

In [ ]:
from huggingface_hub import list_repo_files
print([f for f in list_repo_files("yandex/yambda", repo_type="dataset") if "mapping" in f])


In [ ]:
from datasets import load_dataset

def load_map(fname):
    return load_dataset("yandex/yambda", data_files=fname)["train"].to_pandas()

am = load_map("artist_item_mapping.parquet")
bm = load_map("album_item_mapping.parquet")
print(am.columns.tolist(), am.shape)
print(bm.columns.tolist(), bm.shape)

maps = {}
for gcol, mp in [("artist_id", am), ("album_id", bm)]:
    src = [c for c in mp.columns if c != "item_id"][0]
    maps[gcol] = mp.drop_duplicates("item_id").set_index("item_id")[src]
    L[gcol] = L.item_id.map(maps[gcol]).fillna(-1).astype("int64")
    K[gcol] = K.item_id.map(maps[gcol]).fillna(-1).astype("int64")
    print(gcol, "покрытие маппингом:", (L[gcol] >= 0).mean())

def add_indirect(X, cutoff, windows=WINDOWS):
    X = X.drop(columns=[c for c in X.columns if c.startswith(("uar_", "ual_")) or c in ("artist_id", "album_id")],
               errors="ignore")
    HL = L[L.timestamp < cutoff]
    HK = K[K.timestamp < cutoff]
    for gname, gcol in [("ar", "artist_id"), ("al", "album_id")]:
        X[gcol] = X.item_id.map(maps[gcol]).fillna(-1).astype("int64")
        for w in windows:
            lo = 0 if w is None else cutoff - w * DAY
            wn = "all" if w is None else f"{w}d"
            for ev, h in [("listens", HL[HL.timestamp >= lo]), ("likes", HK[HK.timestamp >= lo])]:
                col = f"u{gname}_{ev}_{wn}"
                c = h.groupby(["uid", gcol]).size().rename(col).reset_index()
                X = X.merge(c, on=["uid", gcol], how="left")
                X[col] = X[col].fillna(0)
            X[f"u{gname}_cr_{wn}"] = X[f"u{gname}_likes_{wn}"] / (X[f"u{gname}_listens_{wn}"] + 1)
    return X

def build_train2(n_cuts=4, step=7):
    parts = []
    for k in range(1, n_cuts + 1):
        c = t_val - k * step * DAY
        X = build(window(c, c + step * DAY), c)
        X = X[X.ui_likes_all == 0]
        parts.append(add_indirect(X, c))
    return pd.concat(parts, ignore_index=True)

t0 = time.time()
Xtr = build_train2();          print("train", Xtr.shape, time.time() - t0)
Xva = add_indirect(Xva, t_val);  Xte = add_indirect(Xte, t_test)
print(Xva.shape, Xte.shape, time.time() - t0)

In [ ]:
import warnings; warnings.filterwarnings("ignore")

# 0) сохранить таблицы, чтобы не пересчитывать после перезапуска ядра
import os
os.makedirs("data", exist_ok=True)
os.makedirs("reports", exist_ok=True)
print("рабочая папка:", os.getcwd())

Xtr.to_parquet("data/Xtr.parquet"); Xva.to_parquet("data/Xva.parquet"); Xte.to_parquet("data/Xte.parquet")
ID = ("uid", "item_id", "label", "artist_id", "album_id")
IND = ("uar_", "ual_")
base         = ["is_organic"] + [c for c in Xtr.columns if c.endswith("_all") and c not in ID and not c.startswith(IND)]
feats_manual = [c for c in Xtr.columns if c not in ID and not c.startswith(IND)]
feats_ind    = [c for c in Xtr.columns if c not in ID]
print(len(base), len(feats_manual), len(feats_ind), "признаков")
print("доли позитивов:", Xtr.label.mean(), Xva.label.mean(), Xte.label.mean())

rows = []
def log(name, out):
    for nm in ("val", "test"):
        rows.append({"model": name, "split": nm, **out[nm]})

# 1) бейзлайны на отфильтрованных данных
rng = np.random.default_rng(1)
for nm, X in [("val", Xva), ("test", Xte)]:
    r = report(f"random [{nm}]", X, rng.random(len(X)));            rows.append({"model": "random", "split": nm, **r})
    r = report(f"top-popular [{nm}]", X, X.i_likes_all.values);     rows.append({"model": "top-popular", "split": nm, **r})

# 2) три бустинга
m_min, r = fit_eval("minimal (all-time only)", base);              log("minimal (all-time only)", r)
m_man, r = fit_eval("full manual", feats_manual);                  log("full manual", r)
m_ind, r = fit_eval("full manual + artist/album", feats_ind);      log("full manual + artist/album", r)

res = pd.DataFrame(rows).round(4)
print(res.to_string(index=False))
res.to_csv("reports/results_manual.csv", index=False)

imp = pd.Series(m_ind.booster_.feature_importance("gain"), index=feats_ind).sort_values(ascending=False)
print(imp.head(15))